# Grover's Algorithm

Grover's Algorithm is one of the many Quantum Algorithms which is widely used for its quadratic speedup over classical algorithms. It was first designed to solve the unstructured search problem, by Lov Grover in 1996. 

## What does it do?

Suppose you have about 1 million phone number entries from which you need to find one specific phone number. Classically, it is a very tedious task because we need to go through atleast half of the total number of entries one by one on an average. Grover's Algorithm can do that with just a 1000 checks, which is a quadratic speedup.

## How does it work?

We make use of a function, $f(x)$ that returns zero for every other case except for our target input

Let us take an example of searching for a target qudit from a bunch of them. It uses the following three steps-

1. **Superposition**- It puts all the qubits into a superposition using H gates, so that all of them have an equal probability. For instance, with 2 qubits, we get four possibilities $(00,01,10,11)$
2. **Oracle**- This is the brilliance of the algorithm. The Oracle is responsible for marking the correct answer by flipping its phase, using the Z gate. This makes the target input to have a negative amplitude, whereas all the other inputs have a positive amplitude. But measuring now, will only result in a random output.
3. **Diffuser(_Amplitude Amplification_)**- The diffuser is responsible for calculating the average probability of all the inputs. It then reflects back each input state around the average probability. The target state will have a **negative amplitude** and will be **below the average**, so it will be reflected **above** the average. The rest unmarked states will have a **positive amplitude** and will lie **above the average** due to which they will be reflected down **below** the average. By this, the target state will lie above the other states. After $\sqrt N$ iterations, (where $N$ is the total number of possibilities) the probability of the target state will increase to $100\%$

Here is an easier example to imagine.

Let's consider we have two qubits and our target state is say $|11\rangle$. Using H gates, we create superposition and now have four different possible states as $|00\rangle,|01\rangle,|10\rangle,|11\rangle$, each with an equal probability of around $25\%$ and thus have a probability amplitude of $\frac{1}{2}$ to detect. 

Now the oracle marks the target state $|11\rangle$ and due to the phase flip it now has a negative amplitude. After this, the diffuser now calculates the average amplitude and reflects each state based on its amplitude with respect to the average.

The amplitudes of all the four states are now- $|00\rangle= 1/2,|01\rangle= 1/2,|10\rangle= 1/2,|11\rangle= -1/2$. The average amplitude calculated by the diffuser is thus $\frac{\frac{1}{2}+\frac{1}{2}+\frac{1}{2}+\frac{-1}{2}}{4}=\frac{1}{4}$. Since all the remaining unmarked states have an amplitude above the average, the are reflected down the average, except for the target state whose amplitude is below the average and hence will be reflected above.

This makes-

1. $|00\rangle=\frac{1}{2}$ above average hence reflected down to $0$.
2. $|01\rangle=\frac{1}{2}$ above average hence reflected down to $0$.
3. $|10\rangle=\frac{1}{2}$ above average hence reflected down to $0$.
4. $|11\rangle=\frac{-1}{2}$ below average hence reflected above to $1$.

Now our target state has a probability of $|1|^2=100\%$. 

## The circuit components-

1. **H-Gate**- This is to induce superposition so that all the possible states have equal probability to be detected with.
2. **Oracle**- uses X, CZ or multiple Z gates to flip the phase of the target state. The specific gates depend on which state we are making.
3. **Diffuser**- H gates, X gates, a multi-controlled Z gate, then X gates and H gates again. It's a fixed structure that doesn't change regardless of the target.

In [1]:
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator

In [3]:
circuit=QuantumCircuit(2,2)

n=input("Hi! This is to simulate Grover`s Algorithm. Please enter the target state that we want to find using two qubits.:")

#Superposition

circuit.h(0)
circuit.h(1)

#Oracle

if n=="00":
    circuit.x(0)
    circuit.x(1)
    circuit.cz(0,1)  #the cz gate is also called a CPhase gate
    circuit.x(0)
    circuit.x(1)
elif n=="01":
    circuit.x(0)
    circuit.cz(0,1)
    circuit.x(0)
elif n=="10":
    circuit.x(1)
    circuit.cz(0,1)
    circuit.x(1)
elif n=="11":
    circuit.cz(0,1)
else:
    print("Please enter a valid 2-qubit target state using 0s and 1s")

#Diffuser

circuit.h(0)
circuit.h(1)

circuit.x(0)
circuit.x(1)

circuit.cz(0,1)

circuit.x(0)
circuit.x(1)

circuit.h(0)
circuit.h(1)

circuit.measure(0,0)
circuit.measure(1,1)

simulator=AerSimulator()
job=simulator.run(circuit, shots=1000)
result=job.result()
counts=result.get_counts()

measured = list(counts.keys())[0][::-1]
print(f"Target state: {n}")
print(f"Grover's found: {measured}")
print(f"Shots: {list(counts.values())[0]}/1000")

Hi! This is to simulate Grover`s Algorithm. Please enter the target state that we want to find using two qubits.: 11


Target state: 11
Grover's found: 11
Shots: 1000/1000


This was a simple form of the algorithm being implemented for two qubits and finding a target from four possible states. Now let us move on and find a target amongst more number of possible states, say for three qubits, where there are 8 different possible states.

In [1]:
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator

circuit=QuantumCircuit(3,3)

n=input("Please enter the target state for three qubits using 0s and 1s:")

#Superposition

circuit.h(0)
circuit.h(1)
circuit.h(2)

#Oracle

if n=="000":
    circuit.x(0)
    circuit.x(1)
    circuit.x(2)
    circuit.h(2)
    circuit.ccx(0,1,2)  #we do not have CCZ in Qiskit, so we deploy a CCX gate and Hadamard for the phase flip.
    circuit.h(2)
    circuit.x(0)
    circuit.x(1)
    circuit.x(2)
elif n=="001":
    circuit.x(0)
    circuit.x(1)
    circuit.h(2)
    circuit.ccx(0,1,2)  #Based on the structure of the target state, we need to X wrap the appropriate qubits.
    circuit.h(2)        #One thing to note is that irrespective of the target state, the position of the CCX gate is always fixed for the last qubit.
    circuit.x(0)
    circuit.x(1)
elif n=="010":
    circuit.x(0)
    circuit.x(2)
    circuit.h(2)
    circuit.ccx(0,1,2)
    circuit.h(2)
    circuit.x(0)
    circuit.x(2)
elif n=="100":
    circuit.x(1)
    circuit.x(2)
    circuit.h(2)
    circuit.ccx(0,1,2)
    circuit.h(2)
    circuit.x(1)
    circuit.x(2)
elif n=="110":
    circuit.x(2)
    circuit.h(2)
    circuit.ccx(0,1,2)
    circuit.h(2)
    circuit.x(2)
elif n=="101":
    circuit.x(1)
    circuit.h(2)
    circuit.ccx(0,1,2)
    circuit.h(2)
    circuit.x(1)
elif n=="011":
    circuit.x(0)
    circuit.h(2)
    circuit.ccx(0,1,2)
    circuit.h(2)
    circuit.x(0)
elif n=="111":
    circuit.h(2)
    circuit.ccx(0,1,2)
    circuit.h(2)
else:
    print("Please enter a valid target state for three qubits")

#Diffuser

circuit.h(0)
circuit.h(1)
circuit.h(2)
circuit.x(0)
circuit.x(1)
circuit.x(2)
circuit.h(2)
circuit.ccx(0,1,2)
circuit.h(2)
circuit.x(0)
circuit.x(1)
circuit.x(2)
circuit.h(0)
circuit.h(1)
circuit.h(2)

circuit.measure(0,0)
circuit.measure(1,1)
circuit.measure(2,2)

simulator=AerSimulator()
job=simulator.run(circuit, shots=1000)
result=job.result()
counts=result.get_counts()

measured = list(counts.keys())[0][::-1]
print(f"Target state: {n}")
print(f"Grover's found: {measured}")
print(f"Shots: {list(counts.values())[0]}/1000")

Please enter the target state for three qubits using 0s and 1s: 111


Target state: 111
Grover's found: 111
Shots: 772/1000


As we can see that for greater number of qubits, the probability drops a little from $100\%$ for the target state for one round of running the algorithm. We can increase this by running it for couple of times as shown below.

In [1]:
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator

circuit=QuantumCircuit(3,3)

n=input("Please enter the target state for three qubits using 0s and 1s:")

#Superposition

circuit.h(0)
circuit.h(1)
circuit.h(2)

for i in range(2):
    #Oracle
    if n=="000":
        circuit.x(0)
        circuit.x(1)
        circuit.x(2)
        circuit.h(2)
        circuit.ccx(0,1,2)  #we do not have CCZ in Qiskit, so we deploy a CCX gate and Hadamard for the phase flip.
        circuit.h(2)
        circuit.x(0)
        circuit.x(1)
        circuit.x(2)
    elif n=="001":
        circuit.x(0)
        circuit.x(1)
        circuit.h(2)
        circuit.ccx(0,1,2)  #Based on the structure of the target state, we need to X wrap the appropriate qubits.
        circuit.h(2)        #One thing to note is that irrespective of the target state, the position of the CCX gate is always fixed for the last qubit.
        circuit.x(0)
        circuit.x(1)
    elif n=="010":
        circuit.x(0)
        circuit.x(2)
        circuit.h(2)
        circuit.ccx(0,1,2)
        circuit.h(2)
        circuit.x(0)
        circuit.x(2)
    elif n=="100":
        circuit.x(1)
        circuit.x(2)
        circuit.h(2)
        circuit.ccx(0,1,2)
        circuit.h(2)
        circuit.x(1)
        circuit.x(2)
    elif n=="110":
        circuit.x(2)
        circuit.h(2)
        circuit.ccx(0,1,2)
        circuit.h(2)
        circuit.x(2)
    elif n=="101":
        circuit.x(1)
        circuit.h(2)
        circuit.ccx(0,1,2)
        circuit.h(2)
        circuit.x(1)
    elif n=="011":
        circuit.x(0)
        circuit.h(2)
        circuit.ccx(0,1,2)
        circuit.h(2)
        circuit.x(0)
    elif n=="111":
        circuit.h(2)
        circuit.ccx(0,1,2)
        circuit.h(2)
    else:
        print("Please enter a valid target state for three qubits")

    #Diffuser

    circuit.h(0)
    circuit.h(1)
    circuit.h(2)
    circuit.x(0)
    circuit.x(1)
    circuit.x(2)
    circuit.h(2)
    circuit.ccx(0,1,2)
    circuit.h(2)
    circuit.x(0)
    circuit.x(1)
    circuit.x(2)
    circuit.h(0)
    circuit.h(1)
    circuit.h(2)

circuit.measure(0,0)
circuit.measure(1,1)
circuit.measure(2,2)

simulator=AerSimulator()
job=simulator.run(circuit, shots=1000)
result=job.result()
counts=result.get_counts()

measured = list(counts.keys())[0][::-1]
print(f"Target state: {n}")
print(f"Grover's found: {measured}")
print(f"Shots: {list(counts.values())[0]}/1000")

Please enter the target state for three qubits using 0s and 1s: 011


Target state: 011
Grover's found: 011
Shots: 939/1000


As you can see, by introducing a for loop and running it two times has significantly improved the number of shots from 772 to 939. An important result to note is that we can not simply keep on increasing the number of iterations freely to get 1000/1000 shots. For example, if we run the above loop for 3 times, the probability collapses and it starts showing random states.

Consider our quantum state (representing as a 2D vector) to be laying on a 2D plane, where the axes represent correct and incorrect answers. After the superposition, our quantum state is pointing majorly towards the incorrect axis because the probability of the correct state is less than that of the incorrect states in total. For every iteration, the vector keeps getting rotated by an angle $\theta=\frac{2}{\sqrt N}$ per iteration, where $N$ represents total number of possible states. For one iteration, the vector lies closest to the correct axis. This gives us the highest probability for the correct state. But, if we repeat the iterations beyond this, the vector will shoot past this correct axis and will again lie towards the incorrect axis. The correct number of iterations is given as $\frac{\pi}{4}*\sqrt N$.